# Multiscale 1D RM-CLEAN

Standard (Hogbom) RM-CLEAN models the Faraday dispersion function (FDF) as a sum
of delta functions. For a **Faraday-thick** source, whose FDF is genuinely
extended in Faraday depth, that becomes a picket fence of deltas: it still fits
the data, but the model is a spray of components that also soaks up noise.
Multiscale RM-CLEAN (Cornwell 2008; Offringa & Smirnov 2017) cleans with
extended kernels, so an extended feature is modelled by a few broad components
instead.

Two things decide whether multiscale can help:

- **Resolution**: the Faraday thickness $\Delta\phi$ must exceed the RMSF FWHM,
  else the source is a point as far as CLEAN can tell.
- **Recoverability**: $\Delta\phi$ must be below the largest recoverable scale
  $\phi_{\rm max} = \pi / \lambda^2_{\rm min}$ (set by the highest frequency),
  else the structure is resolved out and no algorithm recovers it.

We run the same three regimes (thin, marginal, resolved-thick) on two very
different coverages:

| Coverage | Bands | RMSF FWHM | $\phi_{\rm max}$ |
|---|---|---|---|
| **RACS-all** | 3 narrow bands (0.8-1.8 GHz) | coarse (~34 rad m$^{-2}$) | only ~3x FWHM: little room |
| **GMIMS-DRAGONS** | continuous 0.3-1.8 GHz | fine (~6 rad m$^{-2}$) | ~19x FWHM: lots of room |

For each thick slab we also clean **two Faraday-thin components at the same
separation**, to check that multiscale adapts to the source: extended for a
genuine slab, tighter for two points.

The value multiscale buys is a model **matched to the source** (extended for a
slab, not a picket fence of deltas). On flux it is roughly comparable to
single-scale, but not identical: a Burn slab genuinely depolarises (the $Q,U$
signal shrinks as $\mathrm{sinc}(\Delta\phi\,\lambda^2)$), so a thick source's
observed flux really is lower and CLEAN cannot invent what depolarisation
removed; meanwhile on *marginally* resolved or closely blended sources
multiscale tends to **over-recover** the integrated flux (the scale kernels are
not orthogonal, so a source thrown onto a scale wider than itself is
double-counted). We quantify this in the cases below and revisit it in the
summary.

**On counting.** These runs have no major cycle (no invert-and-subtract against
the data), so there are two counts. Single-scale places one delta per **minor
iteration** (`n_iter`). Multiscale's `n_iter` is the number of **minor cycles**
(scale re-selections), each running an inner per-scale Hogbom loop; its total
component-placement count is `n_sub_minor_iter`. Multiscale reaches convergence
in far fewer *minor cycles*, but the honest work comparison is
`n_sub_minor_iter` against single-scale's `n_iter`, and by that measure
multiscale places **more** components in these (realistic, sidelobe-rich)
RMSFs, not fewer. So we quote both and do not sell multiscale as "faster": its
value is the model, not the component count.

## Setup

In [ ]:
from __future__ import annotations

import logging

import astropy.units as u
import matplotlib.pyplot as plt
import numpy as np
from numpy.typing import NDArray
from rm_lite.tools_1d import rmclean, rmsynth
from rm_lite.utils.logging import quiet_logs
from rm_lite.utils.synthesis import freq_to_lambda2

plt.rcParams["figure.dpi"] = 110
rng = np.random.default_rng(7)

# Source and CLEAN constants, shared by every case.
RM_RADM2 = 25.0  # true Faraday depth of the source
PSI0_DEG = 10.0  # intrinsic polarisation angle
FRAC_POL = 0.5  # intrinsic (peak) fractional polarisation
RMS_NOISE = 0.02  # noise per Q, U channel
AUTO_MASK = 10.0  # mask threshold, in units of the FDF noise
AUTO_THRESHOLD = 0.5  # clean threshold in FDF noise units; the mask lets us go deep
MULTISCALE_MAX_ITER_SUB_MINOR = 3000
SCALE_BIAS = 0.6  # library default (WSClean's); keeps a thin source on the delta scale
SCALE_BIAS_HIGH = 0.9  # near 1: scale 0 almost always wins (~ single-scale)
SCALE_BIAS_LOW = 0.4  # too low: over-extends even a thin source

# Faraday thickness of each case, in units of the RMSF FWHM. RACS-all can only
# reach ~3x FWHM, so its "resolved" case is milder than GMIMS-DRAGONS' (~19x).
DELTA_THIN = 0.0
DELTA_MARGINAL = 1.0
DELTA_RESOLVED_RACS = 2.0
DELTA_RESOLVED_GMIMS = 6.0


def burn_slab(
    lambda_sq_arr_m2: NDArray[np.float64], delta_rm_radm2: float
) -> NDArray[np.complex128]:
    """Burn slab P(lambda^2); delta_rm=0 is a Faraday-thin source."""
    return (
        FRAC_POL
        * np.exp(2j * (np.deg2rad(PSI0_DEG) + RM_RADM2 * lambda_sq_arr_m2))
        * np.sinc(delta_rm_radm2 * lambda_sq_arr_m2 / np.pi)
    ).astype(np.complex128)


def two_thin_points(
    lambda_sq_arr_m2: NDArray[np.float64], separation_radm2: float
) -> NDArray[np.complex128]:
    """Two Faraday-thin components at RM +/- separation/2, same total flux."""
    return (
        FRAC_POL
        * np.exp(2j * (np.deg2rad(PSI0_DEG) + RM_RADM2 * lambda_sq_arr_m2))
        * np.cos(separation_radm2 * lambda_sq_arr_m2)
    ).astype(np.complex128)


def clean_stats(res) -> tuple[float, int, int]:
    """(mom0, minor iterations, component steps) for a 1D clean result.

    Single-scale places one component per minor iteration, so the two counts are
    equal. Multiscale's minor count is scale re-selections; `n_sub_minor_iter` is
    the total per-scale Hogbom steps, i.e. the count directly comparable to a
    single-scale iteration count.
    """
    mom0 = float(res.fdf_parameters["mom0_debias"][0])
    minor = int(np.ravel(res.clean_parameters["n_iter"])[0])
    comps = int(np.ravel(res.clean_parameters["n_sub_minor_iter"])[0])
    return mom0, minor, comps

One plot function, a 2x2 grid comparing single-scale (left) and multiscale
(right). The top row shows the FDF (true, dirty, clean); the bottom row shows the
CLEAN model and residual with the mask and threshold levels. The true FDF is a
delta for a thin source, a tophat for a slab, two deltas for the two-point case.</cell id="3">

In [ ]:
def plot_clean(synth, single, multi, source_kind, extent, rmsf_fwhm, title):
    """2x2 comparison. Columns: single-scale vs multiscale. Top row: the FDF
    (true, dirty, clean). Bottom row: the CLEAN model, residual, and the mask and
    threshold levels. Multiscale's minor-cycle and sub-minor totals are annotated
    separately."""
    phi = synth.fdf_arrs["phi_arr_radm2"].to_numpy().astype(float)
    dirty = synth.fdf_arrs["fdf_dirty_complex_arr"].to_numpy().astype(complex)

    # True (intrinsic) Faraday dispersion function: delta / tophat / two deltas.
    true_model = np.zeros_like(phi)
    if source_kind == "thin":
        true_model[np.argmin(np.abs(phi - RM_RADM2))] = FRAC_POL
    elif source_kind == "slab":
        true_model[np.abs(phi - RM_RADM2) <= extent / 2] = FRAC_POL
    elif source_kind == "points":
        for centre in (RM_RADM2 - extent / 2, RM_RADM2 + extent / 2):
            true_model[np.argmin(np.abs(phi - centre))] = FRAC_POL / 2

    half = max(4 * rmsf_fwhm, 0.8 * extent)
    fig, axs = plt.subplots(2, 2, sharex=True, sharey="row", figsize=(10, 6))
    for col, (res, name) in enumerate(
        zip((single, multi), ("single-scale", "multiscale"), strict=True)
    ):
        clean = res.fdf_arrs["fdf_clean_complex_arr"].to_numpy().astype(complex)
        model = res.fdf_arrs["fdf_model_complex_arr"].to_numpy().astype(complex)
        resid = res.fdf_arrs["fdf_residual_complex_arr"].to_numpy().astype(complex)
        mask = float(res.clean_parameters["mask"][0])
        threshold = float(res.clean_parameters["threshold"][0])
        _, minor, comps = clean_stats(res)
        count = (
            f"{minor} iters."
            if name == "single-scale"
            else f"{minor} iters.,\n{comps} sub-minor"
        )

        # Top row: the FDF.
        top = axs[0, col]
        top.plot(phi, true_model, color="0.5", ls="--", lw=1.2, label="true FDF")
        top.plot(phi, np.abs(dirty), color="k", alpha=0.35, label="dirty")
        top.plot(phi, np.abs(clean), color="k", label="clean")
        top.step(phi, np.abs(model), color="tab:red", where="mid", label="model")
        top.plot(phi, np.abs(resid), color="tab:blue", alpha=0.6, label="residual")
        top.axhline(mask, color="tab:orange", ls=":", lw=0.8, label="mask")
        top.axhline(threshold, color="tab:green", ls=":", lw=0.8, label="threshold")
        top.text(
            0.02,
            0.95,
            f"{name}: {count}",
            transform=top.transAxes,
            va="top",
            fontweight="bold",
        )
        top.set(ylabel="|FDF|")

        # Bottom row: model, residual, mask and threshold.
        bot = axs[1, col]
        bot.step(phi, np.abs(model), color="tab:red", where="mid", label="model")
        bot.plot(phi, np.abs(resid), color="tab:blue", alpha=0.6, label="residual")
        bot.axhline(mask, color="tab:orange", ls=":", lw=0.8, label="mask")
        bot.axhline(threshold, color="tab:green", ls=":", lw=0.8, label="threshold")
        bot.set(
            ylabel="|model|, |residual|",
            xlabel=rf"$\phi$ / {u.rad / u.m**2:latex_inline}",
            xlim=(RM_RADM2 - half, RM_RADM2 + half),
        )
    axs[0, 0].legend(fontsize=7, loc="upper right")
    axs[1, 0].legend(fontsize=7, loc="upper right")
    fig.suptitle(title)
    fig.tight_layout()

## RACS-all coverage

RACS observes three narrow bands (RACS-low, -mid, -high). The gaps make the
$\lambda^2$ coverage sparse, so the RMSF is broad (coarse Faraday resolution)
and, because the highest frequency is only ~1.8 GHz, the largest recoverable
scale is barely a few RMSF FWHM. There is little room for genuinely resolved
thick structure.

In [ ]:
bw_low, bw_mid, bw_high = 288, 144, 288  # MHz
freq_hz = (
    np.concatenate(
        [
            np.linspace(943.5 - bw_low / 2, 943.5 + bw_low / 2, 36),
            np.linspace(1367.5 - bw_mid / 2, 1367.5 + bw_mid / 2, 9),
            np.linspace(1655.5 - bw_high / 2, 1655.5 + bw_high / 2, 9),
        ]
    )
    * u.MHz
)
freq_hz = freq_hz.to(u.Hz).value
lambda_sq = freq_to_lambda2(freq_hz)

### Thin source ($\Delta\phi = 0$)

The thin case also fixes the RMSF FWHM and $\phi_{\rm max}$ we scale the other
cases against. A point source has nothing extended to recover, so multiscale
should match single-scale on both flux and shape, and it does: at the default
`scale_bias=0.6` the point stays on the delta scale. The auto scale grid puts
its first extended scale well wider than the RMSF (following WSClean, the first
nonzero scale has a shape FWHM ~1.8x the RMSF FWHM), so convolving a point with
any extended scale drops its peak enough that the delta scale wins. See the
`scale_bias` section near the end for what sets this balance.

In [ ]:
pol = (
    burn_slab(lambda_sq, DELTA_THIN)
    + rng.normal(0, RMS_NOISE, freq_hz.size)
    + 1j * rng.normal(0, RMS_NOISE, freq_hz.size)
).astype(np.complex128)
err = np.ones_like(pol) * (RMS_NOISE + 1j * RMS_NOISE)
with quiet_logs(logging.ERROR):
    synth = rmsynth.run_rmsynth(freq_hz, pol, err, n_samples=20, do_fit_rmsf=True)
    single = rmclean.run_rmclean_from_synth(
        synth, auto_mask=AUTO_MASK, auto_threshold=AUTO_THRESHOLD
    )
    multi = rmclean.run_rmclean_from_synth(
        synth,
        auto_mask=AUTO_MASK,
        auto_threshold=AUTO_THRESHOLD,
        multiscale=True,
        multiscale_scale_bias=SCALE_BIAS,
        multiscale_max_iter_sub_minor=MULTISCALE_MAX_ITER_SUB_MINOR,
    )

single_mom0, single_n, _ = clean_stats(single)
multi_mom0, multi_minor, multi_comps = clean_stats(multi)

rmsf_fwhm = float(synth.fdf_parameters["fwhm_rmsf_radm2"][0])
phi_max = float(synth.fdf_parameters["phi_max_scale_radm2"][0])
print(f"RACS-all RMSF FWHM = {rmsf_fwhm:.1f} rad/m^2")
print(
    f"max recoverable scale = {phi_max:.0f} rad/m^2 ({phi_max / rmsf_fwhm:.1f} x FWHM)"
)
print(
    f"marginal case dRM = {DELTA_MARGINAL * rmsf_fwhm:.0f} rad/m^2, "
    f"resolved case dRM = {DELTA_RESOLVED_RACS * rmsf_fwhm:.0f} rad/m^2 "
    f"({DELTA_RESOLVED_RACS * rmsf_fwhm / phi_max:.0%} of max scale)"
)

print(f"single: mom0={single_mom0:.3f} iterations={single_n}")
print(
    f"multi:  mom0={multi_mom0:.3f} minor_cycles={multi_minor} "
    f"component_steps={multi_comps}"
)

# Multiscale matches single-scale flux on the point source. (It reaches it in
# few minor cycles but many component steps; see "On counting" above.)
assert abs(multi_mom0 - single_mom0) < 0.05
assert 0.40 < multi_mom0 < 0.60

In [ ]:
plot_clean(synth, single, multi, "thin", 0.0, rmsf_fwhm, "RACS-all: thin")

### Marginally resolved ($\Delta\phi \approx 1\times$ FWHM)

In [ ]:
delta_rm = DELTA_MARGINAL * rmsf_fwhm

pol = (
    burn_slab(lambda_sq, delta_rm)
    + rng.normal(0, RMS_NOISE, freq_hz.size)
    + 1j * rng.normal(0, RMS_NOISE, freq_hz.size)
).astype(np.complex128)
err = np.ones_like(pol) * (RMS_NOISE + 1j * RMS_NOISE)
with quiet_logs(logging.ERROR):
    synth = rmsynth.run_rmsynth(freq_hz, pol, err, n_samples=20, do_fit_rmsf=True)
    single = rmclean.run_rmclean_from_synth(
        synth, auto_mask=AUTO_MASK, auto_threshold=AUTO_THRESHOLD
    )
    multi = rmclean.run_rmclean_from_synth(
        synth,
        auto_mask=AUTO_MASK,
        auto_threshold=AUTO_THRESHOLD,
        multiscale=True,
        multiscale_scale_bias=SCALE_BIAS,
        multiscale_max_iter_sub_minor=MULTISCALE_MAX_ITER_SUB_MINOR,
    )

single_mom0, single_n, _ = clean_stats(single)
multi_mom0, multi_minor, multi_comps = clean_stats(multi)

print(f"dRM={delta_rm:.0f} rad/m^2 ({DELTA_MARGINAL:.0f}x FWHM)")
print(f"single: mom0={single_mom0:.3f} iterations={single_n}")
print(
    f"multi:  mom0={multi_mom0:.3f} minor_cycles={multi_minor} "
    f"component_steps={multi_comps}"
)

# Depolarised source; multiscale recovers comparable flux.
assert multi_mom0 > 0.75 * single_mom0

In [ ]:
plot_clean(synth, single, multi, "slab", delta_rm, rmsf_fwhm, "RACS-all: marginal")

### Resolved thick ($\Delta\phi \approx 2\times$ FWHM)

About as thick as RACS-all can usefully go: the slab is over half the largest
recoverable scale and heavily depolarised. Multiscale does not recover more flux
here (the structure is near the edge of the usable window), but still describes
it as a few broad features rather than a picket fence of deltas.

In [ ]:
delta_rm = DELTA_RESOLVED_RACS * rmsf_fwhm

pol = (
    burn_slab(lambda_sq, delta_rm)
    + rng.normal(0, RMS_NOISE, freq_hz.size)
    + 1j * rng.normal(0, RMS_NOISE, freq_hz.size)
).astype(np.complex128)
err = np.ones_like(pol) * (RMS_NOISE + 1j * RMS_NOISE)
with quiet_logs(logging.ERROR):
    synth = rmsynth.run_rmsynth(freq_hz, pol, err, n_samples=20, do_fit_rmsf=True)
    single = rmclean.run_rmclean_from_synth(
        synth, auto_mask=AUTO_MASK, auto_threshold=AUTO_THRESHOLD
    )
    multi = rmclean.run_rmclean_from_synth(
        synth,
        auto_mask=AUTO_MASK,
        auto_threshold=AUTO_THRESHOLD,
        multiscale=True,
        multiscale_scale_bias=SCALE_BIAS,
        multiscale_max_iter_sub_minor=MULTISCALE_MAX_ITER_SUB_MINOR,
    )

single_mom0, single_n, _ = clean_stats(single)
multi_mom0, multi_minor, multi_comps = clean_stats(multi)

print(
    f"dRM={delta_rm:.0f} rad/m^2 ({DELTA_RESOLVED_RACS:.0f}x FWHM, "
    f"{delta_rm / phi_max:.0%} of max scale)"
)
print(f"single: mom0={single_mom0:.3f} iterations={single_n}")
print(
    f"multi:  mom0={multi_mom0:.3f} minor_cycles={multi_minor} "
    f"component_steps={multi_comps}"
)

# Heavily depolarised at the edge of the usable window; flux still comparable.
assert multi_mom0 > 0.7 * single_mom0

In [ ]:
plot_clean(
    synth, single, multi, "slab", delta_rm, rmsf_fwhm, "RACS-all: resolved thick"
)

### Same separation, two thin points

Two Faraday-thin components separated by the same $\Delta\phi$ as the slab above.
Unlike the slab they do not depolarise, so single-scale and multiscale both
recover the full flux; multiscale still does it in far fewer iterations. The
contrast with the slab (same separation, but the slab's flux collapses) shows
that on RACS it is Faraday *thickness*, not separation, that is the problem.

In [ ]:
pol = (
    two_thin_points(lambda_sq, delta_rm)
    + rng.normal(0, RMS_NOISE, freq_hz.size)
    + 1j * rng.normal(0, RMS_NOISE, freq_hz.size)
).astype(np.complex128)
err = np.ones_like(pol) * (RMS_NOISE + 1j * RMS_NOISE)
with quiet_logs(logging.ERROR):
    synth = rmsynth.run_rmsynth(freq_hz, pol, err, n_samples=20, do_fit_rmsf=True)
    single = rmclean.run_rmclean_from_synth(
        synth, auto_mask=AUTO_MASK, auto_threshold=AUTO_THRESHOLD
    )
    multi = rmclean.run_rmclean_from_synth(
        synth,
        auto_mask=AUTO_MASK,
        auto_threshold=AUTO_THRESHOLD,
        multiscale=True,
        multiscale_scale_bias=SCALE_BIAS,
        multiscale_max_iter_sub_minor=MULTISCALE_MAX_ITER_SUB_MINOR,
    )

single_mom0, single_n, _ = clean_stats(single)
multi_mom0, multi_minor, multi_comps = clean_stats(multi)

print(f"two thin points separated by {delta_rm:.0f} rad/m^2")
print(f"single: mom0={single_mom0:.3f} iterations={single_n}")
print(
    f"multi:  mom0={multi_mom0:.3f} minor_cycles={multi_minor} "
    f"component_steps={multi_comps}"
)

# Two points do not depolarise: both recover near-full flux.
assert multi_mom0 > 0.8 * FRAC_POL

In [ ]:
plot_clean(
    synth,
    single,
    multi,
    "points",
    delta_rm,
    rmsf_fwhm,
    "RACS-all: two thin points (same separation)",
)

**RACS-all takeaway:** narrow-band coverage leaves almost no room for resolved
thick sources, and they depolarise quickly, so multiscale buys no extra flux for
a slab. Two thin points at the same separation are recovered fine by both, which
pins the blame on thickness, not separation. Multiscale converges in fewer
*minor cycles* but (see the component-step counts) does not do less work.</cell id="0">

## GMIMS-DRAGONS coverage

GMIMS-DRAGONS-style coverage is continuous from 0.3 to 1.8 GHz. The dense, wide
$\lambda^2$ span gives a fine RMSF (good Faraday resolution) and a largest
recoverable scale of roughly 19 RMSF FWHM, so genuinely resolved thick structure
both exists and survives.

In [ ]:
freq_hz = np.linspace(0.3e9, 1.8e9, 500)
lambda_sq = freq_to_lambda2(freq_hz)

### Thin source ($\Delta\phi = 0$)

In [ ]:
pol = (
    burn_slab(lambda_sq, DELTA_THIN)
    + rng.normal(0, RMS_NOISE, freq_hz.size)
    + 1j * rng.normal(0, RMS_NOISE, freq_hz.size)
).astype(np.complex128)
err = np.ones_like(pol) * (RMS_NOISE + 1j * RMS_NOISE)
with quiet_logs(logging.ERROR):
    synth = rmsynth.run_rmsynth(freq_hz, pol, err, n_samples=20, do_fit_rmsf=True)
    single = rmclean.run_rmclean_from_synth(
        synth, auto_mask=AUTO_MASK, auto_threshold=AUTO_THRESHOLD
    )
    multi = rmclean.run_rmclean_from_synth(
        synth,
        auto_mask=AUTO_MASK,
        auto_threshold=AUTO_THRESHOLD,
        multiscale=True,
        multiscale_scale_bias=SCALE_BIAS,
        multiscale_max_iter_sub_minor=MULTISCALE_MAX_ITER_SUB_MINOR,
    )

single_mom0, single_n, _ = clean_stats(single)
multi_mom0, multi_minor, multi_comps = clean_stats(multi)

rmsf_fwhm = float(synth.fdf_parameters["fwhm_rmsf_radm2"][0])
phi_max = float(synth.fdf_parameters["phi_max_scale_radm2"][0])
print(f"GMIMS-DRAGONS RMSF FWHM = {rmsf_fwhm:.1f} rad/m^2")
print(
    f"max recoverable scale = {phi_max:.0f} rad/m^2 ({phi_max / rmsf_fwhm:.1f} x FWHM)"
)
print(
    f"resolved case dRM = {DELTA_RESOLVED_GMIMS * rmsf_fwhm:.0f} rad/m^2 "
    f"({DELTA_RESOLVED_GMIMS * rmsf_fwhm / phi_max:.0%} of max scale)"
)

print(f"single: mom0={single_mom0:.3f} iterations={single_n}")
print(
    f"multi:  mom0={multi_mom0:.3f} minor_cycles={multi_minor} "
    f"component_steps={multi_comps}"
)

assert abs(multi_mom0 - single_mom0) < 0.05
assert 0.40 < multi_mom0 < 0.60

# Keep the thin-source results to compare against an over-aggressive scale_bias.
synth_thin, multi_thin = synth, multi

In [ ]:
plot_clean(synth, single, multi, "thin", 0.0, rmsf_fwhm, "GMIMS-DRAGONS: thin")

### Marginally resolved ($\Delta\phi \approx 1\times$ FWHM)

In [ ]:
delta_rm = DELTA_MARGINAL * rmsf_fwhm

pol = (
    burn_slab(lambda_sq, delta_rm)
    + rng.normal(0, RMS_NOISE, freq_hz.size)
    + 1j * rng.normal(0, RMS_NOISE, freq_hz.size)
).astype(np.complex128)
err = np.ones_like(pol) * (RMS_NOISE + 1j * RMS_NOISE)
with quiet_logs(logging.ERROR):
    synth = rmsynth.run_rmsynth(freq_hz, pol, err, n_samples=20, do_fit_rmsf=True)
    single = rmclean.run_rmclean_from_synth(
        synth, auto_mask=AUTO_MASK, auto_threshold=AUTO_THRESHOLD
    )
    multi = rmclean.run_rmclean_from_synth(
        synth,
        auto_mask=AUTO_MASK,
        auto_threshold=AUTO_THRESHOLD,
        multiscale=True,
        multiscale_scale_bias=SCALE_BIAS,
        multiscale_max_iter_sub_minor=MULTISCALE_MAX_ITER_SUB_MINOR,
    )

single_mom0, single_n, _ = clean_stats(single)
multi_mom0, multi_minor, multi_comps = clean_stats(multi)

print(f"dRM={delta_rm:.0f} rad/m^2 ({DELTA_MARGINAL:.0f}x FWHM)")
print(f"single: mom0={single_mom0:.3f} iterations={single_n}")
print(
    f"multi:  mom0={multi_mom0:.3f} minor_cycles={multi_minor} "
    f"component_steps={multi_comps} (over-recovery {multi_mom0 / single_mom0 - 1:+.0%})"
)

# A source barely wider than the RMSF is thrown onto the first extended scale,
# which is wider than the source; the non-orthogonal kernels then double-count,
# so multiscale over-recovers the flux here (bounded, but real). This is the
# limitation summarised at the end, not a target to reproduce.
assert single_mom0 < multi_mom0 < 1.5 * single_mom0

In [ ]:
plot_clean(synth, single, multi, "slab", delta_rm, rmsf_fwhm, "GMIMS-DRAGONS: marginal")

### Resolved thick ($\Delta\phi \approx 6\times$ FWHM)

Now the source is genuinely resolved and well inside the recoverable window.
Flux is comparable to single-scale (a Burn slab still depolarises, so neither
recovers the full intrinsic flux). The difference is the **model**: in the plot's
bottom row multiscale spreads it across the slab rather than piling deltas at a
point. It gets there in fewer minor cycles, though not fewer component steps.</cell id="30">

In [ ]:
delta_rm = DELTA_RESOLVED_GMIMS * rmsf_fwhm

pol = (
    burn_slab(lambda_sq, delta_rm)
    + rng.normal(0, RMS_NOISE, freq_hz.size)
    + 1j * rng.normal(0, RMS_NOISE, freq_hz.size)
).astype(np.complex128)
err = np.ones_like(pol) * (RMS_NOISE + 1j * RMS_NOISE)
with quiet_logs(logging.ERROR):
    synth = rmsynth.run_rmsynth(freq_hz, pol, err, n_samples=20, do_fit_rmsf=True)
    single = rmclean.run_rmclean_from_synth(
        synth, auto_mask=AUTO_MASK, auto_threshold=AUTO_THRESHOLD
    )
    multi = rmclean.run_rmclean_from_synth(
        synth,
        auto_mask=AUTO_MASK,
        auto_threshold=AUTO_THRESHOLD,
        multiscale=True,
        multiscale_scale_bias=SCALE_BIAS,
        multiscale_max_iter_sub_minor=MULTISCALE_MAX_ITER_SUB_MINOR,
    )

single_mom0, single_n, _ = clean_stats(single)
multi_mom0, multi_minor, multi_comps = clean_stats(multi)

multi_model = np.abs(multi.fdf_arrs["fdf_model_complex_arr"].to_numpy().astype(complex))
multi_model_chan = int((multi_model > 0.01 * multi_model.max()).sum())
print(
    f"dRM={delta_rm:.0f} rad/m^2 ({DELTA_RESOLVED_GMIMS:.0f}x FWHM, "
    f"{delta_rm / phi_max:.0%} of max scale)"
)
print(f"single: mom0={single_mom0:.3f} iterations={single_n}")
print(
    f"multi:  mom0={multi_mom0:.3f} minor_cycles={multi_minor} "
    f"component_steps={multi_comps} (model spans {multi_model_chan} channels)"
)

# Comparable flux, and here is the genuine multiscale behaviour: unlike the thin
# source, the slab model spreads across many channels rather than sitting at a
# point. This is what the model+residual plot below makes visible.
assert multi_mom0 > 0.95 * single_mom0
assert multi_model_chan > 50

In [ ]:
plot_clean(
    synth, single, multi, "slab", delta_rm, rmsf_fwhm, "GMIMS-DRAGONS: resolved thick"
)

### Same separation, two thin points

The same $\Delta\phi$ separation as the resolved slab, but two thin components.
These do not depolarise, so both algorithms recover close to the full flux.
Compare the model (bottom row) with the slab's: multiscale's two-point model is
tighter than its slab model, but it is not two clean deltas: the interference
structure between the two RMSF responses draws in some extended-scale flux, so
the model still spreads and the recovered flux runs a few percent high. This is
the same non-orthogonality limitation as the marginal case, milder here because
the two points are individually unresolved.

In [ ]:
pol = (
    two_thin_points(lambda_sq, delta_rm)
    + rng.normal(0, RMS_NOISE, freq_hz.size)
    + 1j * rng.normal(0, RMS_NOISE, freq_hz.size)
).astype(np.complex128)
err = np.ones_like(pol) * (RMS_NOISE + 1j * RMS_NOISE)
with quiet_logs(logging.ERROR):
    synth = rmsynth.run_rmsynth(freq_hz, pol, err, n_samples=20, do_fit_rmsf=True)
    single = rmclean.run_rmclean_from_synth(
        synth, auto_mask=AUTO_MASK, auto_threshold=AUTO_THRESHOLD
    )
    multi = rmclean.run_rmclean_from_synth(
        synth,
        auto_mask=AUTO_MASK,
        auto_threshold=AUTO_THRESHOLD,
        multiscale=True,
        multiscale_scale_bias=SCALE_BIAS,
        multiscale_max_iter_sub_minor=MULTISCALE_MAX_ITER_SUB_MINOR,
    )

single_mom0, single_n, _ = clean_stats(single)
multi_mom0, multi_minor, multi_comps = clean_stats(multi)

multi_model = np.abs(multi.fdf_arrs["fdf_model_complex_arr"].to_numpy().astype(complex))
multi_model_chan = int((multi_model > 0.01 * multi_model.max()).sum())
print(f"two thin points separated by {delta_rm:.0f} rad/m^2")
print(f"single: mom0={single_mom0:.3f} iterations={single_n}")
print(
    f"multi:  mom0={multi_mom0:.3f} minor_cycles={multi_minor} "
    f"component_steps={multi_comps} (model spans {multi_model_chan} channels)"
)

# Two points do not depolarise: both recover near-full flux.
assert multi_mom0 > 0.85 * FRAC_POL

In [ ]:
plot_clean(
    synth,
    single,
    multi,
    "points",
    delta_rm,
    rmsf_fwhm,
    "GMIMS-DRAGONS: two thin points (same separation)",
)

### How a scale gets selected (and why the grid matters)

Each minor cycle, multiscale convolves the residual with every scale kernel,
takes the peak $p_s$ of each, and selects the scale that maximises $p_s\,S(s)$,
where $S(s)$ is the scale-bias function (Offringa & Smirnov 2017, eq. 3;
$S(0)=1$, larger for larger scales when `scale_bias` $< 1$). The kernels are
unit-integral, so convolution is a weighted average and $p_s \le p_0$ *always*:
the delta scale has the highest raw peak for any source. A larger scale wins
only when its bias boost $S(s)$ beats the peak drop $p_0/p_s$.

That makes the whole thing hinge on how much an extended scale drops a *point
source's* peak. If the first extended scale sits almost on top of the RMSF, it
barely drops a point's peak, so the bias trivially steals point sources onto a
gaussian and there is no `scale_bias` that both keeps points compact and cleans
real structure. The fix is the scale grid, not the bias: put the first extended
scale well wider than the RMSF (WSClean's rule, a shape FWHM ~1.8x the beam),
and a point drops enough to stay on the delta scale while a genuinely extended
source still prefers a gaussian. Below we show the selected scale for a true
point vs a genuine scale-4 structure, on the default grid and on a deliberately
too-fine one.

In [ ]:
from rm_lite.utils.clean import (
    convolve_fdf_scale,
    make_scales,
    scale_bias_function,
)

# Work in the GMIMS-DRAGONS RMSF (from the thin case above).
rmsf_c = synth_thin.rmsf_arrs["rmsf_complex_arr"].to_numpy().astype(complex)
phi2_c = synth_thin.rmsf_arrs["phi2_arr_radm2"].to_numpy().astype(float)
fwhm_c = float(synth_thin.fdf_parameters["fwhm_rmsf_radm2"][0])
KERNEL = "tapered_quad"

# Two ground-truth "dirty" residuals whose scale we know:
# - a true point source appears in the dirty FDF as the RMSF itself;
# - a genuine scale-4 structure appears as (RMSF convolved with the scale-4 kernel).
delta_resid = rmsf_c / np.abs(rmsf_c).max()
scale4_resid = np.asarray(
    convolve_fdf_scale(4.0, fwhm_c, rmsf_c, phi2_c, KERNEL), complex
)
scale4_resid = scale4_resid / np.abs(scale4_resid).max()


def scale_peaks(resid, scales):
    """Peak of resid convolved with each scale kernel (the p_s of O&S Alg. 1)."""
    return np.array(
        [
            np.nanmax(
                np.abs(convolve_fdf_scale(float(s), fwhm_c, resid, phi2_c, KERNEL))
            )
            for s in scales
        ]
    )


def winning_scale(resid, scales, bias):
    """Scale that maximises p_s * S(s): the scale O&S would select."""
    p = scale_peaks(resid, scales)
    return float(scales[int(np.argmax(p * scale_bias_function(scales, bias)))])


# The auto grid (default first_scale=4: first extended scale ~1.8x RMSF FWHM)
# vs a too-fine grid (first_scale=1) that sits almost on top of the RMSF.
grids = {
    "current (first_scale=4)": make_scales(20.0, first_scale=4.0),
    "too-fine (first_scale=1)": make_scales(20.0, first_scale=1.0),
}
for name, scales in grids.items():
    row = ", ".join(
        f"bias {b}: scale {winning_scale(delta_resid, scales, b):g}"
        for b in (0.6, 0.7, 0.8)
    )
    print(f"{name:26s} | true delta   -> {row}")
    row = ", ".join(
        f"bias {b}: scale {winning_scale(scale4_resid, scales, b):g}"
        for b in (0.6, 0.7, 0.8)
    )
    print(f"{name:26s} | scale-4 src  -> {row}")

default_grid = grids["current (first_scale=4)"]
toofine_grid = grids["too-fine (first_scale=1)"]
# On the default grid a true point stays on the delta scale, while a genuinely
# extended source engages a gaussian. The too-fine grid steals the point onto a
# gaussian (which is the bug the grid change fixes).
assert winning_scale(delta_resid, default_grid, SCALE_BIAS) == 0.0
assert winning_scale(scale4_resid, default_grid, SCALE_BIAS) > 0.0
assert winning_scale(delta_resid, toofine_grid, SCALE_BIAS) > 0.0

# Plot the selection score p_s * S(s) (normalised to the delta-scale peak) at the
# default bias: the delta peaks at scale 0, the scale-4 source peaks at scale 4.
sb = scale_bias_function(default_grid, SCALE_BIAS)
fig, ax = plt.subplots(figsize=(6, 3.2))
for resid, label, color in (
    (delta_resid, "true point source", "tab:blue"),
    (scale4_resid, "scale-4 structure", "tab:red"),
):
    p = scale_peaks(resid, default_grid)
    score = p * sb / p[0]
    ax.plot(default_grid, score, "o-", color=color, label=label)
    ax.plot(default_grid[np.argmax(score)], score.max(), "k*", ms=12, zorder=5)
ax.axhline(1.0, color="0.6", lw=0.6)
ax.set(
    xlabel="scale (RMSF FWHM units)",
    ylabel="selection score  $p_s\\,S(s)$  (norm. to scale 0)",
    title=f"Scale selection at bias {SCALE_BIAS} (star = winner)",
)
ax.legend(fontsize=8)
fig.tight_layout()

### The `scale_bias` tradeoff

With the scale grid set relative to the RMSF (above), `multiscale_scale_bias`
(Offringa & Smirnov 2017, eq. 3) is the fine control: how strongly larger scales
are preferred once a source is extended enough to warrant them. It does *not*
have to be traded off against keeping point sources compact, that job belongs to
the grid. The regimes:

- **0.9** (near 1): $S(s)\approx1$ for all scales, so scale 0 almost always
  wins and multiscale reduces to single-scale, losing the extended-model
  behaviour.
- **0.6** (our default, WSClean's): the extended scales engage on genuinely
  resolved structure, while a point source still stays on the delta scale.
- **0.4** (too low): the bias boost is large enough to over-extend even a thin
  source, inflating its flux and broadening its peak.

We re-clean the thin GMIMS source at all three. A thin source should stay
compact from 0.9 down to the default 0.6, and only broaden once the bias is
pushed too low.

In [ ]:
phi = synth_thin.fdf_arrs["phi_arr_radm2"].to_numpy().astype(float)
biases = {
    "0.9 (~single-scale)": SCALE_BIAS_HIGH,
    "0.6 (default)": SCALE_BIAS,
    "0.4 (too low)": SCALE_BIAS_LOW,
}
cleans, widths, mom0s = {}, {}, {}
with quiet_logs(logging.ERROR):
    for label, bias in biases.items():
        res = rmclean.run_rmclean_from_synth(
            synth_thin,
            auto_mask=AUTO_MASK,
            auto_threshold=AUTO_THRESHOLD,
            multiscale=True,
            multiscale_scale_bias=bias,
            multiscale_max_iter_sub_minor=MULTISCALE_MAX_ITER_SUB_MINOR,
        )
        cl = np.abs(res.fdf_arrs["fdf_clean_complex_arr"].to_numpy().astype(complex))
        cleans[label] = cl
        widths[label] = int((cl > cl.max() / 2).sum())
        mom0s[label] = float(res.fdf_parameters["mom0_debias"][0])
        print(
            f"scale_bias={bias}: mom0={mom0s[label]:.3f} half-power width={widths[label]} chan"
        )

# From near-1 down to the default, a thin source stays compact (same flux and
# half-power width): the corrected grid keeps it on the delta scale. Only when
# the bias is pushed too low does it over-extend, inflating flux and broadening.
assert np.isclose(mom0s["0.6 (default)"], mom0s["0.9 (~single-scale)"], atol=0.02)
assert widths["0.6 (default)"] == widths["0.9 (~single-scale)"]
assert mom0s["0.4 (too low)"] > mom0s["0.6 (default)"]
assert widths["0.4 (too low)"] > widths["0.6 (default)"]

fig, ax = plt.subplots(figsize=(6, 3))
for label, color in zip(biases, ("tab:blue", "tab:green", "tab:red"), strict=True):
    ax.plot(phi, cleans[label], color=color, label=label)
ax.axvline(RM_RADM2, color="0.5", ls=":", lw=1, label="true RM")
ax.set(
    xlabel=rf"$\phi$ / {u.rad / u.m**2:latex_inline}",
    ylabel="|clean FDF|",
    xlim=(RM_RADM2 - 4 * rmsf_fwhm, RM_RADM2 + 4 * rmsf_fwhm),
)
ax.legend(fontsize=8)
fig.tight_layout()

## Summary

| Regime | $\Delta\phi$ / FWHM | Multiscale vs single-scale |
|---|---|---|
| Thin | 0 | matches flux, stays a compact delta at the default bias |
| Marginal | ~1 | **over-recovers flux** (source thrown onto a too-wide scale) |
| Resolved thick | few (if coverage allows) | comparable flux, model spread across the slab |
| Two thin points | any separation | close to full flux, model tighter than a slab but not two clean deltas |

The key points:

- **Flux is not conserved exactly.** On a point source (or a genuinely resolved
  thick one) multiscale matches single-scale flux. But because the scale kernels
  are **not orthogonal**, a source thrown onto a scale wider than itself is
  double-counted, so on marginally resolved or closely blended sources
  multiscale **over-recovers** the integrated flux (a few percent for two blended
  points, up to ~20% for a ~1-FWHM slab). Treat the recovered flux of
  barely-resolved features with caution. Separately, a Burn slab genuinely
  depolarises, so a thick source's observed flux is physically lower and no CLEAN
  variant recovers it.
- **Not fewer components.** Multiscale converges in fewer *minor cycles*, but in
  realistic (sidelobe-rich) RMSFs it places *more* component-placement steps than
  single-scale, not fewer.
- **The genuine value is the model:** source-matched (extended for a slab,
  compact for a point) rather than a picket fence of deltas. Detection is set by
  the coverage (dense low-$\lambda^2$ / high frequency), not by the algorithm.
- **Scale grid before bias.** Keeping point sources on the delta scale is the
  grid's job (first extended scale ~1.8x the RMSF FWHM, WSClean's rule);
  `scale_bias` is the fine control for how eagerly extended structure engages the
  larger scales. The two-phase auto-mask confines each scale to where it was
  detected, which removes spurious far-flung components but does not cure the
  flux over-recovery above.

Enable multiscale with `multiscale=True` on `run_rmclean_from_synth` (1D) or
`rmclean_3d_from_synth` (3D). Give it a generous $\phi$ window (scale kernels
wider than the window pick up edge artefacts); the mask confines cleaning to
significant channels, so a deep threshold (these runs use $0.5\sigma$) is safe.
Tune with `multiscale_scale_bias` (default 0.6; towards 1 reduces to
single-scale, below ~0.5 over-extends compact sources), `multiscale_scales`
(explicit scales if the auto grid does not suit), `multiscale_max_iter_sub_minor`,
and the usual `auto_mask` / `auto_threshold`.